# 6380 T1D Selection1 stLearn Analysis with H&E + CCI/LR Analysis

Notebook prepared for the **6380 T1D Xenium Selection1 ROI**.

Main parts:

1. Set paths and output folders  
2. Load ROI cells and Xenium cell metadata  
3. Create ROI AnnData from `cell_feature_matrix.h5`  
4. H&E alignment / plotting using rotation `k=1`  
5. Normalize, PCA, UMAP, Louvain clustering  
6. Run stLearn PSTS / pseudotime  
7. Plot high-definition H&E pseudotime  
8. Detect transition markers  
9. Prepare CCI object  
10. Run LR permutation test and downstream CCI analysis  

Output folder used in the code:

```text
/media/mayurdoke/Xtra-Drive/stlearn 6380 T1D
```

In [ ]:
# ============================================================
# CELL 0) SETUP
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import stlearn as st
import matplotlib.pyplot as plt
import matplotlib as mpl
import tifffile

from pathlib import Path
from scipy import sparse
from io import StringIO
from matplotlib.lines import Line2D

plt.rcParams["font.family"] = mpl.rcParamsDefault["font.family"]
plt.rcParams["font.sans-serif"] = mpl.rcParamsDefault["font.sans-serif"]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# ------------------------------------------------------------
# Input folder
# ------------------------------------------------------------
XENIUM_DIR = Path(
    "/media/mayurdoke/Xtra-Drive/Stlearn_spatialVelocity/ROI_exports_UPDATED/"
    "output-XETG00394__6263_6380_6393__6380_Pan_body_02__20250801__182140"
)

HE_FILE = XENIUM_DIR / "6380_H&E_FIXED.ome.tif"
CELL_FEATURE_MATRIX = XENIUM_DIR / "cell_feature_matrix.h5"
CELLS_CSV_GZ = XENIUM_DIR / "cells.csv.gz"
ROI_CELLS_FILE = XENIUM_DIR / "Selection_1_cells_stats.csv"
ROI_COORD_FILE = XENIUM_DIR / "Selection_1_coordinates.csv"

LIBRARY_ID = "Xenium_6380_T1D_Selection1"

# ------------------------------------------------------------
# Output folders
# ------------------------------------------------------------
OUTDIR = Path("/media/mayurdoke/Xtra-Drive/stlearn 6380 T1D")
FIG_DIR = OUTDIR / "01_figures"
TABLE_DIR = OUTDIR / "02_tables"
PSTS_DIR = OUTDIR / "03_PSTS_trajectory"
CCI_DIR = OUTDIR / "04_CCI_LR_analysis"
OBJECT_DIR = OUTDIR / "05_objects"

for d in [OUTDIR, FIG_DIR, TABLE_DIR, PSTS_DIR, CCI_DIR, OBJECT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTDIR)

print("\nFile check:")
for name, path in {
    "H&E": HE_FILE,
    "cell_feature_matrix.h5": CELL_FEATURE_MATRIX,
    "cells.csv.gz": CELLS_CSV_GZ,
    "Selection_1_cells_stats.csv": ROI_CELLS_FILE,
    "Selection_1_coordinates.csv": ROI_COORD_FILE
}.items():
    print(f"{name:35s}", path.exists(), path)

In [ ]:
# ============================================================
# CELL 1) ROBUSTLY READ SELECTION1 ROI CELL-STATS FILE
# ============================================================

def robust_read_roi_cells(path):
    path = str(path)
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        raw_lines = [line.rstrip("\n\r") for line in f if line.strip()]

    print("Total non-empty lines:", len(raw_lines))
    print("First raw lines:")
    for i, line in enumerate(raw_lines[:8]):
        print(i, ":", line)

    header_idx = None
    for i, line in enumerate(raw_lines):
        low = line.lower()
        if ("cell id" in low or "cell_id" in low or "cellid" in low) and "cluster" in low:
            header_idx = i
            break

    if header_idx is None:
        for i, line in enumerate(raw_lines):
            low = line.lower()
            if ("cell id" in low or "cell_id" in low or "cellid" in low):
                header_idx = i
                break

    if header_idx is None:
        raise ValueError("Could not find ROI cell table header.")

    data_text = "\n".join(raw_lines[header_idx:])

    for sep in [",", "\t", ";", "|"]:
        try:
            df = pd.read_csv(StringIO(data_text), sep=sep, engine="python", on_bad_lines="skip")
            df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
            cols_low = [c.lower() for c in df.columns]
            has_cell = any(c in cols_low for c in ["cell id", "cell_id", "cellid", "cell"])
            if df.shape[1] > 1 and has_cell:
                print("Parsed with separator:", repr(sep))
                return df
        except Exception:
            pass

    raise ValueError("Could not parse ROI cell stats file.")


roi_cells = robust_read_roi_cells(ROI_CELLS_FILE)
print("\nROI cells parsed:", roi_cells.shape)
print(roi_cells.columns.tolist())
display(roi_cells.head())

# Standardize cell IDs
cell_col = None
for c in ["Cell ID", "cell_id", "cell", "CellID", "Cell Id", "barcode"]:
    if c in roi_cells.columns:
        cell_col = c
        break

if cell_col is None:
    for c in roi_cells.columns:
        if c.lower().replace(" ", "").replace("_", "") in ["cellid", "cell"]:
            cell_col = c
            break

if cell_col is None:
    raise ValueError("Could not identify cell ID column.")

roi_cells["cell_id"] = roi_cells[cell_col].astype(str).str.strip()

# Standardize cluster labels
cluster_col = None
for c in roi_cells.columns:
    if c.lower().strip() == "cluster":
        cluster_col = c
        break

if cluster_col is not None:
    roi_cells["Xenium_cluster"] = (
        roi_cells[cluster_col]
        .astype(str)
        .str.replace("Cluster ", "", regex=False)
        .str.strip()
    )
else:
    roi_cells["Xenium_cluster"] = "NA"

selection1_cell_ids = set(roi_cells["cell_id"].astype(str))

print("\nSelection1 unique cell count:", len(selection1_cell_ids))
print("\nSelection1 Xenium clusters:")
print(roi_cells["Xenium_cluster"].value_counts())

roi_cells.to_csv(TABLE_DIR / "6380_Selection1_ROI_cells_loaded.csv", index=False)

In [ ]:
# ============================================================
# CELL 2) LOAD cells.csv.gz AND MATCH SELECTION1 CELLS
# ============================================================

cells_full = pd.read_csv(CELLS_CSV_GZ)
cells_full.columns = cells_full.columns.str.strip()

print("Full cells metadata:", cells_full.shape)
print(cells_full.columns.tolist())
display(cells_full.head())

if "cell_id" not in cells_full.columns:
    raise ValueError("Expected 'cell_id' in cells.csv.gz.")

cells_full["cell_id"] = cells_full["cell_id"].astype(str).str.strip()

cells_roi = cells_full[cells_full["cell_id"].isin(selection1_cell_ids)].copy()

print("\nCells matched to Selection1:", cells_roi.shape[0], "out of", len(selection1_cell_ids))

if cells_roi.shape[0] == 0:
    raise ValueError("No Selection1 cells matched cells.csv.gz.")

cluster_map = dict(zip(roi_cells["cell_id"].astype(str), roi_cells["Xenium_cluster"].astype(str)))
cells_roi["Xenium_cluster"] = cells_roi["cell_id"].map(cluster_map)

if "x_centroid" not in cells_roi.columns or "y_centroid" not in cells_roi.columns:
    raise ValueError("Expected x_centroid/y_centroid in cells.csv.gz.")

# Store raw Xenium coordinates as imagecol/imagerow initially
cells_roi["imagecol"] = cells_roi["x_centroid"].astype(float)
cells_roi["imagerow"] = cells_roi["y_centroid"].astype(float)

print("\nSelection1 Xenium cluster counts:")
print(cells_roi["Xenium_cluster"].value_counts())

print("\nCoordinate summary:")
display(cells_roi[["imagecol", "imagerow"]].describe())

cells_roi.to_csv(TABLE_DIR / "6380_Selection1_cells_full_metadata_matched.csv", index=False)

In [ ]:
# ============================================================
# CELL 3) LOAD XENIUM EXPRESSION MATRIX AND CREATE ROI AnnData
# ============================================================

import h5py

try:
    adata_xenium = sc.read_10x_h5(CELL_FEATURE_MATRIX, gex_only=False)
    print("Loaded with scanpy.read_10x_h5")
except Exception as e:
    print("scanpy.read_10x_h5 failed. Manual reading.")
    print("Error:", e)

    with h5py.File(CELL_FEATURE_MATRIX, "r") as f:
        mat = f["matrix"]
        data = mat["data"][:]
        indices = mat["indices"][:]
        indptr = mat["indptr"][:]
        shape = mat["shape"][:]

        barcodes = [
            x.decode("utf-8") if isinstance(x, bytes) else str(x)
            for x in mat["barcodes"][:]
        ]

        features = mat["features"]
        gene_names = [
            x.decode("utf-8") if isinstance(x, bytes) else str(x)
            for x in features["name"][:]
        ]

        X = sparse.csc_matrix((data, indices, indptr), shape=tuple(shape)).T.tocsr()

    adata_xenium = sc.AnnData(X=X)
    adata_xenium.obs_names = barcodes
    adata_xenium.var_names = gene_names

adata_xenium.obs_names = adata_xenium.obs_names.astype(str).str.strip()
adata_xenium.obs_names_make_unique()
adata_xenium.var_names = adata_xenium.var_names.astype(str).str.upper()
adata_xenium.var_names_make_unique()

roi_ids = set(cells_roi["cell_id"].astype(str))
matched_ids = [cid for cid in adata_xenium.obs_names if cid in roi_ids]

print("Expression cells matched to Selection1:", len(matched_ids), "out of", len(roi_ids))

if len(matched_ids) == 0:
    raise ValueError("No ROI cells matched the expression matrix.")

adata_he = adata_xenium[matched_ids, :].copy()

# Attach metadata
cells_meta = cells_roi.drop_duplicates("cell_id").set_index("cell_id").loc[adata_he.obs_names].copy()
for col in cells_meta.columns:
    adata_he.obs[col] = cells_meta[col].values

adata_he.obs["cell_id"] = adata_he.obs_names.astype(str)
adata_he.obs["Selection1_selected"] = True
adata_he.obs["Selection1_inside_polygon"] = True
adata_he.obs["Xenium_cluster"] = adata_he.obs["Xenium_cluster"].astype(str)

adata_he.obsm["spatial"] = adata_he.obs[["imagecol", "imagerow"]].to_numpy(dtype=float)
adata_he.layers["counts"] = adata_he.X.copy()

# Minimal spatial metadata for stLearn
adata_he.uns["spatial"] = {
    LIBRARY_ID: {
        "images": {},
        "scalefactors": {
            "tissue_hires_scalef": 1.0,
            "spot_diameter_fullres": 1.0,
        },
        "metadata": {
            "source_image_path": str(HE_FILE),
            "library_id": LIBRARY_ID,
        },
    }
}

# QC
if sparse.issparse(adata_he.X):
    adata_he.obs["n_counts"] = np.asarray(adata_he.X.sum(axis=1)).ravel()
    adata_he.obs["n_genes"] = np.asarray((adata_he.X > 0).sum(axis=1)).ravel()
else:
    adata_he.obs["n_counts"] = adata_he.X.sum(axis=1)
    adata_he.obs["n_genes"] = (adata_he.X > 0).sum(axis=1)

print("\nROI AnnData:")
print(adata_he)
print("\nXenium clusters:")
print(adata_he.obs["Xenium_cluster"].value_counts())

raw_roi_file = OUTDIR / "6380_T1D_Selection1_ROI_raw_light.h5ad"
adata_he.write_h5ad(raw_roi_file)
print("Saved:", raw_roi_file)

## Important H&E Coordinate Note

In the earlier 6380 workflow, the H&E overlay was correct using `rotation k=1`. This plotting function applies:

```python
x_he = imagerow * scale_y
y_he = lvl_w - imagecol * scale_x
```

If your H&E overlay is off, replace `adata_he.obs["imagecol"]` and `adata_he.obs["imagerow"]` with the verified stLearn-aligned coordinates from your prior working object, then rerun this cell and everything downstream.

In [ ]:
# ============================================================
# CELL 4) H&E ALIGNMENT CHECK
# ============================================================

HE_LEVEL = 3  # use 2 for sharper H&E if memory allows
ROTATE_HE_K = 1

with tifffile.TiffFile(HE_FILE) as tif:
    n_levels = len(tif.series[0].levels)
    level_use = min(HE_LEVEL, n_levels - 1)
    he_img = tif.series[0].levels[level_use].asarray()
    full_h, full_w = tif.series[0].levels[0].shape[:2]

lvl_h, lvl_w = he_img.shape[:2]
scale_x = lvl_w / full_w
scale_y = lvl_h / full_h

he_rot = np.rot90(he_img, k=ROTATE_HE_K)
rot_h, rot_w = he_rot.shape[:2]

print("H&E level:", level_use)
print("Full H&E:", (full_h, full_w))
print("Level H&E:", he_img.shape)
print("Rotated H&E:", he_rot.shape)

def add_he_coords_k1(obs_df):
    out = obs_df.copy()
    x0 = out["imagecol"].astype(float).to_numpy() * scale_x
    y0 = out["imagerow"].astype(float).to_numpy() * scale_y
    out["x_he"] = y0
    out["y_he"] = lvl_w - x0
    return out

plot_df = adata_he.obs[
    ["imagecol", "imagerow", "Xenium_cluster", "n_counts", "n_genes"]
].copy()
plot_df = add_he_coords_k1(plot_df)

pad = 100
xmin = max(0, plot_df["x_he"].min() - pad)
xmax = min(rot_w, plot_df["x_he"].max() + pad)
ymin = max(0, plot_df["y_he"].min() - pad)
ymax = min(rot_h, plot_df["y_he"].max() + pad)

crop_xmin = int(np.floor(xmin))
crop_xmax = int(np.ceil(xmax))
crop_ymin = int(np.floor(ymin))
crop_ymax = int(np.ceil(ymax))

he_crop = he_rot[crop_ymin:crop_ymax, crop_xmin:crop_xmax].copy()

plot_df["x_crop"] = plot_df["x_he"] - crop_xmin
plot_df["y_crop"] = plot_df["y_he"] - crop_ymin

xenium_palette = {
    "A11_0": "#ff8c00",
    "A11_1": "#fdbf6f",
    "A6": "#c7e9c0",
    "D1": "#9ecae1",
    "D4": "#3182bd",
    "E1": "#756bb1",
    "Hy4_1": "#b2df8a",
    "Hy4_2": "#9D1FEA",
    "Hy4_3": "#fb9a99",
    "M1": "#a1d99b",
    "Ma3": "#31a354",
    "Msn": "#969696",
    "Polyhormonal endocrine cluster with empty-beta-like features": "#de2d26",
}

xenium_order = [
    "D4", "D1", "A11_0", "A11_1", "Hy4_1", "Hy4_2", "Hy4_3",
    "Polyhormonal endocrine cluster with empty-beta-like features",
    "E1", "M1", "Ma3", "Msn", "A6"
]
xenium_order = [x for x in xenium_order if x in plot_df["Xenium_cluster"].unique()]

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(he_crop, origin="upper", interpolation="lanczos")

for lab in xenium_order:
    sub = plot_df[plot_df["Xenium_cluster"] == lab]
    ax.scatter(
        sub["x_crop"], sub["y_crop"],
        s=42,
        c=xenium_palette.get(lab, "gray"),
        alpha=0.95,
        edgecolors="black",
        linewidths=0.12,
        label=f"{lab} (n={len(sub)})"
    )

ax.set_xlim(0, he_crop.shape[1])
ax.set_ylim(he_crop.shape[0], 0)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("6380 Selection1 H&E alignment check", fontsize=15, fontweight="bold")

ax.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=9,
    markerscale=1.2
)

plt.tight_layout()

out_alignment = FIG_DIR / f"6380_Selection1_HE_alignment_Xenium_labels_level{level_use}.png"
plt.savefig(out_alignment, dpi=600, bbox_inches="tight")
plt.show()

print("Saved:", out_alignment)

In [ ]:
# ============================================================
# CELL 5) NORMALIZATION, PCA, UMAP, LOUVAIN
# ============================================================

roi = adata_he.copy()

if "counts" not in roi.layers:
    roi.layers["counts"] = roi.X.copy()

sc.pp.filter_cells(roi, min_counts=5)
sc.pp.filter_genes(roi, min_cells=3)

sc.pp.normalize_total(roi, target_sum=1e4)
sc.pp.log1p(roi)

sc.pp.highly_variable_genes(
    roi,
    n_top_genes=min(2000, roi.n_vars),
    flavor="seurat"
)

print("HVGs:", int(roi.var["highly_variable"].sum()))

roi_hvg = roi[:, roi.var["highly_variable"]].copy()
sc.pp.scale(roi_hvg, max_value=10)

n_pcs = min(30, roi_hvg.n_obs - 1, roi_hvg.n_vars - 1)
sc.pp.pca(roi_hvg, n_comps=n_pcs, svd_solver="arpack", random_state=0)

roi.obsm["X_pca"] = roi_hvg.obsm["X_pca"]

n_neighbors = min(12, roi.n_obs - 1)

sc.pp.neighbors(
    roi,
    n_neighbors=n_neighbors,
    n_pcs=n_pcs,
    use_rep="X_pca"
)

sc.tl.umap(roi, random_state=0)

try:
    sc.tl.louvain(
        roi,
        resolution=0.7,
        key_added="roi_louvain",
        flavor="igraph"
    )
except Exception:
    sc.tl.louvain(
        roi,
        resolution=0.7,
        key_added="roi_louvain"
    )

roi.obs["roi_louvain"] = roi.obs["roi_louvain"].astype(str)

print("\nLouvain counts:")
print(roi.obs["roi_louvain"].value_counts().sort_index())

print("\nXenium_cluster vs roi_louvain:")
display(pd.crosstab(roi.obs["Xenium_cluster"], roi.obs["roi_louvain"]))

clustered_file = OUTDIR / "6380_T1D_Selection1_ROI_louvain.h5ad"
roi.write_h5ad(clustered_file)
print("Saved:", clustered_file)

In [ ]:
# ============================================================
# CELL 6) RUN stLearn PSTS PSEUDOTIME
# ============================================================

roi_psts = roi.copy()

# stLearn PSTS expects numeric categorical labels
roi_psts.obs["roi_louvain"] = roi_psts.obs["roi_louvain"].astype(str)
roi_psts.obs["roi_louvain"] = pd.Categorical(
    roi_psts.obs["roi_louvain"],
    categories=sorted(roi_psts.obs["roi_louvain"].unique(), key=lambda x: int(x))
)

PSTS_EPS = 200

st.spatial.trajectory.pseudotime(
    roi_psts,
    eps=PSTS_EPS,
    use_rep="X_pca",
    use_label="roi_louvain",
    n_neighbors=12,
    reverse=False
)

print("Pseudotime summary:")
display(roi_psts.obs["dpt_pseudotime"].describe())

print("\nPSTS subcluster counts:")
print(roi_psts.obs["sub_cluster_labels"].value_counts().sort_index())

# Safe save
roi_psts_safe = roi_psts.copy()
roi_psts_safe.uns = {
    "PSTS_analysis_info": {
        "sample": "6380_T1D",
        "roi": "Selection1",
        "eps": PSTS_EPS,
        "use_label": "roi_louvain",
        "use_rep": "X_pca"
    }
}

psts_file = OUTDIR / "6380_T1D_Selection1_PSTS_louvain_based_eps200_SAFE.h5ad"
roi_psts_safe.write_h5ad(psts_file)

roi_psts.obs.to_csv(PSTS_DIR / "Selection1_PSTS_obs_metadata.csv")

print("Saved:", psts_file)

In [ ]:
# ============================================================
# CELL 7) OVERALL PSEUDOTIME ON HIGH-DEFINITION H&E
# ============================================================

roi_psts = sc.read_h5ad(OUTDIR / "6380_T1D_Selection1_PSTS_louvain_based_eps200_SAFE.h5ad")
roi_psts.obs["Xenium_cluster"] = roi_psts.obs["Xenium_cluster"].astype(str)
roi_psts.obs["roi_louvain"] = roi_psts.obs["roi_louvain"].astype(str)
roi_psts.obs["sub_cluster_labels"] = roi_psts.obs["sub_cluster_labels"].astype(str)

HE_LEVEL_ZOOM = 3  # use 2 if memory allows

with tifffile.TiffFile(HE_FILE) as tif:
    n_levels = len(tif.series[0].levels)
    level_use = min(HE_LEVEL_ZOOM, n_levels - 1)
    he_img = tif.series[0].levels[level_use].asarray()
    full_h, full_w = tif.series[0].levels[0].shape[:2]

lvl_h, lvl_w = he_img.shape[:2]
scale_x = lvl_w / full_w
scale_y = lvl_h / full_h

he_rot = np.rot90(he_img, k=1)
rot_h, rot_w = he_rot.shape[:2]

def add_he_coords_k1(obs_df):
    out = obs_df.copy()
    x0 = out["imagecol"].astype(float).to_numpy() * scale_x
    y0 = out["imagerow"].astype(float).to_numpy() * scale_y
    out["x_he"] = y0
    out["y_he"] = lvl_w - x0
    return out

plot_df = roi_psts.obs[
    ["imagecol", "imagerow", "Xenium_cluster", "roi_louvain", "sub_cluster_labels", "dpt_pseudotime"]
].copy()

plot_df = add_he_coords_k1(plot_df)

pad_x = 35
pad_y = 35

xmin = max(0, plot_df["x_he"].min() - pad_x)
xmax = min(rot_w, plot_df["x_he"].max() + pad_x)
ymin = max(0, plot_df["y_he"].min() - pad_y)
ymax = min(rot_h, plot_df["y_he"].max() + pad_y)

crop_xmin = int(np.floor(xmin))
crop_xmax = int(np.ceil(xmax))
crop_ymin = int(np.floor(ymin))
crop_ymax = int(np.ceil(ymax))

he_crop = he_rot[crop_ymin:crop_ymax, crop_xmin:crop_xmax].copy()

plot_df["x_crop"] = plot_df["x_he"] - crop_xmin
plot_df["y_crop"] = plot_df["y_he"] - crop_ymin

plot_crop = plot_df[
    (plot_df["x_crop"] >= 0) &
    (plot_df["x_crop"] <= he_crop.shape[1]) &
    (plot_df["y_crop"] >= 0) &
    (plot_df["y_crop"] <= he_crop.shape[0])
].copy()

# Summaries
pt_by_louvain = (
    plot_crop.groupby("roi_louvain", observed=True)["dpt_pseudotime"]
    .agg(["count", "min", "median", "mean", "max"])
    .sort_index()
)
display(pt_by_louvain)
pt_by_louvain.to_csv(PSTS_DIR / "Selection1_overall_pseudotime_summary_by_louvain.csv")

pt_by_xenium = (
    plot_crop.groupby("Xenium_cluster", observed=True)["dpt_pseudotime"]
    .agg(["count", "min", "median", "mean", "max"])
    .sort_values("median")
)
display(pt_by_xenium)
pt_by_xenium.to_csv(PSTS_DIR / "Selection1_overall_pseudotime_summary_by_Xenium_cluster.csv")

# Plot
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(he_crop, origin="upper", interpolation="lanczos")

scat = ax.scatter(
    plot_crop["x_crop"],
    plot_crop["y_crop"],
    c=plot_crop["dpt_pseudotime"],
    cmap="viridis",
    s=62,
    alpha=0.94,
    linewidths=0
)

ax.set_xlim(0, he_crop.shape[1])
ax.set_ylim(he_crop.shape[0], 0)
ax.set_aspect("equal")
ax.axis("off")

cbar = plt.colorbar(scat, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("Pseudotime", fontsize=11)

plt.tight_layout()

out_pt = PSTS_DIR / f"Selection1_HD_level{level_use}_overall_pseudotime_on_HE_clean.png"
plt.savefig(out_pt, dpi=600, bbox_inches="tight")
plt.show()

print("Saved:", out_pt)

In [ ]:
# ============================================================
# CELL 8) TRANSITION MARKERS
# ============================================================

roi_psts = sc.read_h5ad(OUTDIR / "6380_T1D_Selection1_PSTS_louvain_based_eps200_SAFE.h5ad")
roi_psts.obs["sub_cluster_labels"] = roi_psts.obs["sub_cluster_labels"].astype(str)

branch_auto = (
    roi_psts.obs
    .groupby("sub_cluster_labels", observed=True)["dpt_pseudotime"]
    .median()
    .sort_values()
    .index
    .astype(int)
    .tolist()
)

print("Automatic branch ordered by median pseudotime:")
print(branch_auto)

st.spatial.trajectory.detect_transition_markers_branches(
    roi_psts,
    branch=branch_auto,
    cutoff_spearman=0.1,
    use_raw_count=False
)

transition_keys = [k for k in roi_psts.uns.keys() if k.startswith("branch_")]
print("Transition keys:", transition_keys)

pd.DataFrame({
    "branch_position": range(len(branch_auto)),
    "sub_cluster_label": branch_auto
}).to_csv(PSTS_DIR / "Selection1_transition_marker_branch_used.csv", index=False)

if len(transition_keys) > 0:
    trajectory_key = transition_keys[0]

    st.pl.trajectory.transition_markers_plot(
        roi_psts,
        top_genes=20,
        trajectory=trajectory_key
    )

    plt.tight_layout()

    out_transition_plot = PSTS_DIR / "Selection1_overall_transition_markers_top20.png"
    plt.savefig(out_transition_plot, dpi=350, bbox_inches="tight")
    plt.show()

    print("Saved:", out_transition_plot)

In [ ]:
# ============================================================
# CELL 9) PREPARE CCI OBJECT
# ============================================================

raw_roi_file = OUTDIR / "6380_T1D_Selection1_ROI_raw_light.h5ad"
adata_cci = sc.read_h5ad(raw_roi_file)
adata_cci.obs["cell_id"] = adata_cci.obs_names.astype(str)

roi_psts = sc.read_h5ad(OUTDIR / "6380_T1D_Selection1_PSTS_louvain_based_eps200_SAFE.h5ad")
roi_psts.obs["cell_id"] = roi_psts.obs_names.astype(str)

label_cols = ["Xenium_cluster", "roi_louvain", "sub_cluster_labels", "dpt_pseudotime"]

for col in label_cols:
    label_map = pd.Series(roi_psts.obs[col].values, index=roi_psts.obs_names.astype(str))
    adata_cci.obs[col] = adata_cci.obs_names.astype(str).map(label_map)

print("Missing values after metadata transfer:")
for col in label_cols:
    print(col, int(adata_cci.obs[col].isna().sum()))

adata_cci = adata_cci[~adata_cci.obs["sub_cluster_labels"].isna()].copy()

adata_cci.obs["Xenium_cluster"] = adata_cci.obs["Xenium_cluster"].astype(str)
adata_cci.obs["roi_louvain"] = adata_cci.obs["roi_louvain"].astype(str)
adata_cci.obs["sub_cluster_labels"] = adata_cci.obs["sub_cluster_labels"].astype(str)
adata_cci.obs["dpt_pseudotime"] = adata_cci.obs["dpt_pseudotime"].astype(float)

adata_cci.obs["pseudotime_bin"] = pd.qcut(
    adata_cci.obs["dpt_pseudotime"],
    q=3,
    labels=["Early_PT", "Mid_PT", "Late_PT"],
    duplicates="drop"
).astype(str)

beta_like_label = "Polyhormonal endocrine cluster with empty-beta-like features"

adata_cci.obs["trajectory_stage"] = "Other"
adata_cci.obs.loc[adata_cci.obs["Xenium_cluster"] == "D4", "trajectory_stage"] = "D4 root"
adata_cci.obs.loc[adata_cci.obs["Xenium_cluster"] == "A11_0", "trajectory_stage"] = "A11_0 transition"
adata_cci.obs.loc[adata_cci.obs["Xenium_cluster"] == "Hy4_2", "trajectory_stage"] = "Hy4_2 transition"
adata_cci.obs.loc[adata_cci.obs["Xenium_cluster"] == beta_like_label, "trajectory_stage"] = "Beta-like endpoint"

print("\nPseudotime bin counts:")
print(adata_cci.obs["pseudotime_bin"].value_counts())

print("\nTrajectory-stage counts:")
print(adata_cci.obs["trajectory_stage"].value_counts())

adata_cci.layers["counts"] = adata_cci.X.copy()

sc.pp.filter_cells(adata_cci, min_counts=5)
sc.pp.filter_genes(adata_cci, min_cells=3)

adata_cci.raw = adata_cci.copy()
sc.pp.normalize_total(adata_cci, target_sum=1e4)

# Ensure spatial metadata for stLearn grid
if "spatial" not in adata_cci.uns:
    adata_cci.uns["spatial"] = {
        LIBRARY_ID: {
            "images": {},
            "scalefactors": {
                "tissue_hires_scalef": 1.0,
                "spot_diameter_fullres": 1.0,
            },
            "metadata": {
                "source_image_path": str(HE_FILE),
                "library_id": LIBRARY_ID,
            },
        }
    }

cci_prepared_file = OUTDIR / "6380_T1D_Selection1_CCI_prepared.h5ad"

adata_cci_save = adata_cci.copy()
adata_cci_save.uns = {"spatial": adata_cci.uns["spatial"]}
adata_cci_save.write_h5ad(cci_prepared_file)

adata_cci.obs.to_csv(CCI_DIR / "Selection1_CCI_cell_metadata.csv")

print("Saved:", cci_prepared_file)

In [ ]:
# ============================================================
# CELL 10) BUILD CCI GRID
# ============================================================

cci_prepared_file = OUTDIR / "6380_T1D_Selection1_CCI_prepared.h5ad"
adata_cci = sc.read_h5ad(cci_prepared_file)

# Restore spatial metadata if missing
if "spatial" not in adata_cci.uns:
    adata_cci.uns["spatial"] = {
        LIBRARY_ID: {
            "images": {},
            "scalefactors": {
                "tissue_hires_scalef": 1.0,
                "spot_diameter_fullres": 1.0,
            },
            "metadata": {
                "source_image_path": str(HE_FILE),
                "library_id": LIBRARY_ID,
            },
        }
    }

if "spatial" not in adata_cci.obsm:
    adata_cci.obsm["spatial"] = adata_cci.obs[["imagecol", "imagerow"]].to_numpy(dtype=float)

CCI_LABEL = "pseudotime_bin"

adata_cci.obs[CCI_LABEL] = pd.Categorical(
    adata_cci.obs[CCI_LABEL].astype(str),
    categories=["Early_PT", "Mid_PT", "Late_PT"],
    ordered=True
)

adata_cci.uns[f"{CCI_LABEL}_colors"] = ["#2c7bb6", "#ffffbf", "#d7191c"]

print("CCI label counts:")
print(adata_cci.obs[CCI_LABEL].value_counts())

N_GRID = 20

grid = st.tl.cci.grid(
    adata_cci,
    n_row=N_GRID,
    n_col=N_GRID,
    use_label=CCI_LABEL
)

print("Grid object:")
print(grid)

grid.obs.to_csv(CCI_DIR / f"Selection1_CCI_grid_obs_{CCI_LABEL}.csv")

In [ ]:
# ============================================================
# CELL 11) LR PERMUTATION TEST
# ============================================================

# Compatibility patch for older stLearn + newer numpy
try:
    from numpy.lib._function_base_impl import quantile as _np_quantile_orig
except Exception:
    from numpy.lib.function_base import quantile as _np_quantile_orig

def quantile_compat(a, q, *args, **kwargs):
    if "interpolation" in kwargs and "method" not in kwargs:
        kwargs["method"] = kwargs.pop("interpolation")
    return _np_quantile_orig(a, q, *args, **kwargs)

np.quantile = quantile_compat

lrs = st.tl.cci.load_lrs(
    ["connectomeDB2020_lit"],
    species="human"
)

LR_DISTANCE = 250
N_PAIRS = 500

st.tl.cci.run(
    grid,
    lrs,
    min_spots=3,
    distance=LR_DISTANCE,
    n_pairs=N_PAIRS,
    n_cpus=None
)

print("LR permutation test completed.")
print("grid.uns keys:", list(grid.uns.keys()))

if "lr_summary" in grid.uns:
    display(grid.uns["lr_summary"].head(50))
    grid.uns["lr_summary"].to_csv(CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_summary_raw.csv")

In [ ]:
# ============================================================
# CELL 12) ADJUST P-VALUES + DEFAULT LR SUMMARY PLOT
# ============================================================

st.tl.cci.adj_pvals(
    grid,
    correct_axis="spot",
    pval_adj_cutoff=0.05,
    adj_method="fdr_bh"
)

if "lr_summary" in grid.uns:
    display(grid.uns["lr_summary"].head(50))
    grid.uns["lr_summary"].to_csv(CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_summary_adjusted.csv")

st.pl.lr_summary(
    grid,
    n_top=50,
    figsize=(12, 5)
)

plt.tight_layout()

out_lr_summary = CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_summary_top50.png"
plt.savefig(out_lr_summary, dpi=350, bbox_inches="tight")
plt.show()

print("Saved:", out_lr_summary)

In [ ]:
# ============================================================
# CELL 13) CLEAN LR SUMMARY PLOT WITHOUT YELLOW DOTS
# ============================================================

lr_summary = grid.uns["lr_summary"].copy()
lr_summary["LR_pair"] = lr_summary.index.astype(str)

if "n_spots_sig" in lr_summary.columns:
    y_col = "n_spots_sig"
elif "n_sig_spots" in lr_summary.columns:
    y_col = "n_sig_spots"
elif "n_spots" in lr_summary.columns:
    y_col = "n_spots"
else:
    raise ValueError("Could not find n_spots_sig / n_sig_spots / n_spots column.")

top_n = min(20, lr_summary.shape[0])

plot_lr = (
    lr_summary
    .sort_values(y_col, ascending=False)
    .head(top_n)
    .reset_index(drop=True)
)

plot_lr["rank"] = np.arange(plot_lr.shape[0])

fig, ax = plt.subplots(figsize=(12, 5))

ax.scatter(
    plot_lr["rank"],
    plot_lr[y_col],
    s=45,
    facecolors="white",
    edgecolors="black",
    linewidths=1.2,
    zorder=3
)

for _, row in plot_lr.iterrows():
    ax.text(
        row["rank"],
        row[y_col] + (plot_lr[y_col].max() * 0.025),
        row["LR_pair"],
        rotation=65,
        ha="left",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.set_xlabel(f"LR Rank ({y_col})", fontsize=13, fontweight="bold")
ax.set_ylabel(y_col, fontsize=13, fontweight="bold")

ax.set_xlim(-0.8, plot_lr["rank"].max() + 1.2)
ax.set_ylim(0, plot_lr[y_col].max() * 1.25)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

out_clean_lr = CCI_DIR / "Selection1_CCI_clean_LR_summary_no_yellow_dots.png"
plt.savefig(out_clean_lr, dpi=400, bbox_inches="tight")
plt.show()

print("Saved:", out_clean_lr)

In [ ]:
# ============================================================
# CELL 14) DOWNSTREAM CCI ANALYSIS
# ============================================================

st.tl.cci.run_cci(
    grid,
    CCI_LABEL,
    min_spots=2,
    spot_mixtures=True,
    cell_prop_cutoff=0.1,
    sig_spots=True,
    n_perms=50,
    n_cpus=None
)

print("Downstream CCI completed.")

cci_keys = [k for k in grid.uns.keys() if "cci" in k.lower()]
print("CCI keys:", cci_keys)

if "lr_summary" in grid.uns:
    grid.uns["lr_summary"].to_csv(
        CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_summary_after_run_cci.csv"
    )

for k in cci_keys:
    obj = grid.uns[k]
    print("\nKey:", k, "Type:", type(obj))

    if isinstance(obj, pd.DataFrame):
        display(obj.head())
        obj.to_csv(CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_{k}.csv")

    elif isinstance(obj, dict):
        print("Dictionary length:", len(obj))
        print("First keys:", list(obj.keys())[:10])

cci_key = f"per_lr_cci_{CCI_LABEL}"
if cci_key in grid.uns:
    lr_names = list(grid.uns[cci_key].keys())
    pd.DataFrame({"LR_pair": lr_names}).to_csv(
        CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_available_LR_pair_matrices.csv",
        index=False
    )

print("CCI outputs saved in:", CCI_DIR)

In [ ]:
# ============================================================
# CELL 15) EXTRACT SIGNIFICANT LR PAIRS
# ============================================================

if "lr_summary" not in grid.uns:
    raise ValueError("grid.uns['lr_summary'] not found.")

lr_summary = grid.uns["lr_summary"].copy()

print("LR summary columns:")
print(lr_summary.columns.tolist())
display(lr_summary.head(20))

full_lr_file = CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_summary_full.csv"
lr_summary.to_csv(full_lr_file)

candidate_p_cols = [
    "p_vals_adj", "pval_adj", "adj_pval", "p_adj",
    "p-value adjusted", "pvalue_adj", "p_val_adj", "p.adj"
]

candidate_raw_p_cols = [
    "p_vals", "pval", "p_val", "p-value", "pvalue", "p.val"
]

p_col = next((c for c in candidate_p_cols if c in lr_summary.columns), None)
raw_p_col = next((c for c in candidate_raw_p_cols if c in lr_summary.columns), None)

print("Detected adjusted p-value column:", p_col)
print("Detected raw p-value column:", raw_p_col)

if p_col is not None:
    sig_lr = lr_summary[lr_summary[p_col] < 0.05].copy().sort_values(p_col)
    display(sig_lr.head(50))

    sig_file = CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_significant_LR_pairs_FDR005.csv"
    sig_lr.to_csv(sig_file)
    print("Saved:", sig_file)

elif raw_p_col is not None:
    sig_lr = lr_summary[lr_summary[raw_p_col] < 0.05].copy().sort_values(raw_p_col)
    display(sig_lr.head(50))

    sig_file = CCI_DIR / f"Selection1_CCI_{CCI_LABEL}_LR_pairs_rawP005.csv"
    sig_lr.to_csv(sig_file)
    print("Saved:", sig_file)

else:
    print("No p-value column detected. Saved full LR summary only.")

## How to rerun LR analysis by a different label

For overall pseudotime analysis, the notebook uses:

```python
CCI_LABEL = "pseudotime_bin"
```

To rerun CCI by biological cell labels, modify Cell 10:

```python
CCI_LABEL = "Xenium_cluster"
```

or:

```python
CCI_LABEL = "trajectory_stage"
```

Then rerun Cells 10–15.